In [340]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [ ]:
labeled_df = pd.read_csv('data\labels.csv.')
kyc_people_df = pd.read_csv('data\kyc_individual.csv')
kyc_business_df = pd.read_csv('data\kyc_smallbusiness.csv')
abm_df = pd.read_csv(r'data\abm.csv')
card_df = pd.read_csv('data\card.csv')
cheque_df = pd.read_csv('data\cheque.csv')
eft_df = pd.read_csv('data\eft.csv')
emt_df = pd.read_csv('data\emt.csv')
westernunion_df = pd.read_csv('data\westernunion.csv')
wire_df = pd.read_csv('data\wire.csv')

In [342]:
"""INDIVIDUAL OCCUPATION INCOME BRACKETS"""
#150k+
wage_high = [
    0,        # Legislative and senior management occupations
    10010,    # Financial managers
    10011,    # Human resources managers
    10019,    # Other administrative services managers
    10029,    # Other business services managers
    11201,     # Professional occupations in business management consulting
    20011,   # Architecture and science managers
    20012,   # Computer and information systems managers
    30010,   # Managers in health care
    4001,    # Managers in public administration
    60010,   # Corporate sales managers
    70010,   # Construction managers
    90010    # Manufacturing managers
    ]
#100 - 150k
wage_up_mid = [
    11100,    # Financial auditors and accountants
    11101,    # Financial and investment analysts
    11102,    # Financial advisors
    11202,     # Advertising, marketing and PR professionals
    21222,   # Information systems specialists
    21232,   # Software developers and programmers
    21300,   # Civil engineers
    21301,   # Mechanical engineers
    21310,   # Electrical and electronics engineers
    21321,   # Industrial and manufacturing engineers
    21399,   # Other professional engineers
    2222,    # Technical occupations in computer & info systems
    22301,   # Mechanical engineering technologists & technicians
    31102,   # General practitioners and family physicians
    31202,   # Physiotherapists
    31209,   # Other professional health occupations
    313,     # Nursing & allied health professionals
    3211,    # Technical dental health care
    412,     # Professional occupations in education
    41200,   # University professors
    41220,   # Secondary school teachers
    41221,   # Elementary school & kindergarten teachers
    41101,   # Lawyers & notaries
    21310,   # Electrical & electronics engineers
    52120    # Graphic designers & illustrators
    ]
#60 - 100k
wage_mid = [
    12010,    # Supervisors, general office/admin support
    131,      # Administrative occupations
    13100,    # Administrative officers
    13111,     # Legal administrative assistants
    22220,   # Computer network & web technicians
    22221,   # User support technicians
    32124,   # Pharmacy technicians
    33102,   # Nurse aides / patient service associates
    33109,   # Other assisting occupations in health support
    42100,   # Police officers
    42101,   # Firefighters
    42202,   # Early childhood educators & assistants
    44100,   # Home child care providers
    44101,   # Home support workers / caregivers
    63100,   # Insurance agents / brokers
    63101,   # Real estate agents & salespersons
    62024,   # Cleaning supervisors
    60030,   # Restaurant & food service managers
    6,       # Sales and service occupations (general management)
    9201,    # "Supervisors, processing and manufacturing occupations"
    920,     # "Supervisors, processing, manufacturing, assembly and fabrication occupations"
    62200,   # Chefs
    62100,   # Technical sales specialists - wholesale trade
    831,     # Occupations in natural resources and fisheries
    6004,    # Managers in customer and personal services
    ]
#<35-60k
wage_low_mid = [14200,    # Accounting and related clerks
    14201,     # Banking, insurance and other financial clerks
    63200,   # Cooks
    63202,   # Bakers
    63210,   # Hairstylists / barbers
    641,     # Retail salespersons & non-technical wholesale trade
    64409,   # Customer & information services representatives
    64410,   # Security guards
    6510,    # Cashiers & other sales support
    65100,   # Cashiers
    65102,   # Store shelf stockers, clerks, order fillers
    65200,   # Food & beverage servers
    65201,   # Food counter attendants / kitchen helpers
    6531,    # Cleaners
    72410,   # Automotive service technicians / mechanics
    72310,   # Carpenters
    7230,    # Plumbers, pipefitters, gas fitters
    72200,   # Electricians
    72106,   # Welders / machine operators
    7201,    # Contractors / supervisors in trades
    7410,    # Mail / message distribution
    74102,   # Couriers / messengers
    75110,   # Construction trades helpers / laborers
    75200,   # Taxi / limo drivers
    942,     # Assemblers / inspectors in manufacturing
    4130,    # Social and community service professionals
    73112,   # Painters and decorators (except interior decorators)
    ]

#<35k
wage_low = [14100,    # General office support workers
    14101,    # Receptionists
    14102,     # Personnel clerks
    14402,   # Production logistics workers
    85110,   # Mine laborers
    73300,   # Transport truck drivers
    73301,   # Bus / subway operators
    95109,    # Other laborers in processing / manufacturing / utilities
    ]

wage_low = [str(code) for code in wage_low]
wage_low_mid = [str(code) for code in wage_low_mid]
wage_mid = [str(code) for code in wage_mid]
wage_up_mid = [str(code) for code in wage_up_mid]
wage_high = [str(code) for code in wage_high]

In [343]:
"""BUSINESS REVENUE BRACKETS"""

# $1,000,000+
rev_very_high = [
    "4011",  # Single Family Housing
    "4021",  # Manufacturing and Light Industrial Building
    "4022",  # Commercial Building
    "4112",  # Gas, Oil and Other Energy Related Structures (Except Pipelines)
    "4113",  # Gas and Oil Pipelines
    "4214",  # Excavating and Grading
    "4219"   # Other Site Work
]

# $200,001 - 1,000,000
rev_high = [
    "0919",  # Other Service Industries Incidental to Crude Petroleum and Natural Gas
    "0711",  # Conventional Crude Oil and Natural Gas Industry
    "3081",  # Machine Shop Industry
    "3192",  # Construction and Mining Machinery and Materials Handling Equipment Industry
    "3199",  # Other Machinery and Equipment Industries n.e.c.
    "4214","4224","4226","4229","4231","4232","4233","4235","4239",  # Trades: Concrete, Carpentry, Masonry, Glass, Roofing, Exterior
    "4241","4261","4272","4274","4275","4276","4278","4279",          # Trades: Plumbing, Electrical, Drywall, Finish, Flooring
    "5219","5319","5511","5639"  # Wholesale: Food, Apparel, Automobiles, Building Materials
]

# $50,000 - 200,000
rev_med = [
    # Agriculture & Farming
    "0112","0119","0139","0141","0171","0211","0219","0311","0411","0511","1099","2499",
    # Printing, Publishing, Repair, Machinery
    "2819","2839","3081","3099","3199","3999",
    # Construction (smaller jobs / renovations)
    "4013","4129","4224","4226","4229","4231","4232","4233","4235","4239","4241","4261","4272","4274","4275","4276","4278","4279",
    "4299","4411","4491","4499",
    # Transportation & Utilities
    "4561","4564","4565","4569","4581","4589","4599","4799","4839","4842","4999",
    # Wholesale / Retail
    "5219","5319","5511","5639","5981","5999",
    "6011","6012","6031","6032","6131","6239","6311","6312","6331","6342","6351","6352","6359","6391","6399","6412","6413","6521","6541","6561","6582","6596","6599","6921",
    # Financial, Real Estate, Insurance, Professional Services
    "7214","7215","7292","7421","7511","7512","7599","7611","7711","7712","7721","7722","7731","7739","7741","7742","7749","7751","7752","7759","7761","7771","7791","7799",
    # Health, Education, Social Services
    "8132","8511","8599","8633","8635","8639","8641","8649","8651","8652","8653","8654","8661","8665","8666","8669","8671","8699","8629",
    # Hospitality & Food Services
    "9111","9114","9211","9212","9213","9214",
    # Entertainment, Recreation, Personal Services
    "9611","9631","9639","9641","9659","9699","9711","9712","9713","9726","9741","9799",
    # Organizations & Associations
    "9811","9821","9839","9841","9851","9861",
    # Other services
    "9931","9942","9949","9953","9959","9999","2542","5799","7499","9962"
]

In [344]:
abm_df_subset = abm_df[['customer_id', 'amount_cad', 'debit_credit', 'transaction_datetime']].copy()
abm_df_subset = abm_df_subset.rename(columns={
    'transaction_datetime': 'date'
})
abm_df_subset['transaction_method'] = 'abm'
abm_df_subset = abm_df_subset.dropna(subset=['amount_cad'])
abm_df_subset['date'] = (
    pd.to_datetime(abm_df_subset['date'], utc=True, errors="coerce")
      .dt.tz_convert("America/Toronto")
      .dt.tz_localize(None)
)

card_df_subset = card_df[['customer_id', 'amount_cad', 'debit_credit', 'transaction_datetime']].copy()
card_df_subset = card_df_subset.rename(columns={
    'transaction_datetime': 'date'
})
card_df_subset['transaction_method'] = 'card'
card_df_subset = card_df_subset.dropna(subset=['amount_cad'])
card_df_subset['date'] = (
    pd.to_datetime(card_df_subset['date'], utc=True, errors="coerce")
      .dt.tz_convert("America/Toronto")
      .dt.tz_localize(None)
)

cheque_df_subset = cheque_df[['customer_id', 'amount_cad', 'debit_credit', 'transaction_datetime']].copy()
cheque_df_subset = cheque_df_subset.rename(columns={
    'transaction_datetime': 'date'
})
cheque_df_subset['transaction_method'] = 'cheque'
cheque_df_subset = cheque_df_subset.dropna(subset=['amount_cad'])
cheque_df_subset['date'] = (
    pd.to_datetime(cheque_df_subset['date'], utc=True, errors="coerce")
      .dt.tz_convert("America/Toronto")
      .dt.tz_localize(None)
)

eft_df_subset = eft_df[['customer_id', 'amount_cad', 'debit_credit', 'transaction_datetime']].copy()
eft_df_subset = eft_df_subset.rename(columns={
    'transaction_datetime': 'date'
})
eft_df_subset['transaction_method'] = 'eft'
eft_df_subset = eft_df_subset.dropna(subset=['amount_cad'])
eft_df_subset['date'] = (
    pd.to_datetime(eft_df_subset['date'], utc=True, errors="coerce")
      .dt.tz_convert("America/Toronto")
      .dt.tz_localize(None)
)

emt_df_subset = emt_df[['customer_id', 'amount_cad', 'debit_credit', 'transaction_datetime']].copy()
emt_df_subset = emt_df_subset.rename(columns={
    'transaction_datetime': 'date'
})
emt_df_subset['transaction_method'] = 'emt'
emt_df_subset = emt_df_subset.dropna(subset=['amount_cad'])
emt_df_subset['date'] = (
    pd.to_datetime(emt_df_subset['date'], utc=True, errors="coerce")
      .dt.tz_convert("America/Toronto")
      .dt.tz_localize(None)
)

westernunion_df_subset = westernunion_df[['customer_id', 'amount_cad', 'debit_credit', 'transaction_datetime']].copy()
westernunion_df_subset = westernunion_df_subset.rename(columns={
    'transaction_datetime': 'date'
})
westernunion_df_subset['transaction_method'] = 'westernunion'
westernunion_df_subset = westernunion_df_subset.dropna(subset=['amount_cad'])
westernunion_df_subset['date'] = (
    pd.to_datetime(westernunion_df_subset['date'], utc=True, errors="coerce")
      .dt.tz_convert("America/Toronto")
      .dt.tz_localize(None)
)

wire_df_subset = wire_df[['customer_id', 'amount_cad', 'debit_credit', 'transaction_datetime']].copy()
wire_df_subset = wire_df_subset.rename(columns={
    'transaction_datetime': 'date'
})
wire_df_subset['transaction_method'] = 'wire'
wire_df_subset = wire_df_subset.dropna(subset=['amount_cad'])
wire_df_subset['date'] = (
    pd.to_datetime(wire_df_subset['date'], utc=True, errors="coerce")
      .dt.tz_convert("America/Toronto")
      .dt.tz_localize(None)
)

In [345]:
transactions_long = pd.concat([abm_df_subset, card_df_subset, cheque_df_subset, eft_df_subset, emt_df_subset,
                               westernunion_df_subset, wire_df_subset], ignore_index=True)

person_and_business_df = transactions_long.groupby('customer_id').agg(
    num_transactions = ('amount_cad', 'count'),
    total_amount = ('amount_cad', 'sum'),
    avg_amount = ('amount_cad', 'mean'),
    max_amount = ('amount_cad', 'max'),
    std_amount = ('amount_cad', 'std'),
    num_debit = ('debit_credit', lambda x: (x=='debit').sum()),
    num_credit = ('debit_credit', lambda x: (x=='credit').sum()),
    num_abm = ('transaction_method', lambda x: (x=='abm').sum()),
    num_card = ('transaction_method', lambda x: (x=='card').sum()),
    num_cheque = ('transaction_method', lambda x: (x=='cheque').sum()),
    num_eft = ('transaction_method', lambda x: (x=='eft').sum()),
    num_emt = ('transaction_method', lambda x: (x=='emt').sum()),
    num_westernunion = ('transaction_method', lambda x: (x=='westernunion').sum()),
    num_wire = ('transaction_method', lambda x: (x=='wire').sum())
)

In [346]:
business_ids = set(kyc_business_df['customer_id'])
customer_id = set(kyc_people_df['customer_id'])
person_and_business_df['customer_type'] = np.where(
    person_and_business_df.index.isin(business_ids), 'business',
    np.where(person_and_business_df.index.isin(customer_id), 'person', 'unknown')
)

person_df = person_and_business_df[person_and_business_df['customer_type'] == 'person'].copy()
business_df = person_and_business_df[person_and_business_df['customer_type'] == 'business'].copy()

In [347]:
kyc_people_df.set_index('customer_id', inplace=True, drop=False)

person_df = person_df.join(
    kyc_people_df[['occupation_code','income']],
    how='left'
)

# person_df = person_df.drop(columns = 'customer_type')

In [348]:
person_df
no_data = (((person_df['occupation_code'].isna()) | (person_df['occupation_code'] == 'OTHER' ))& (person_df['income'].isna()))
self_employed = (person_df['occupation_code'] == 'SELF_EMPLOYED') & (person_df['income'].isna())
mask_student = person_df['occupation_code'] == 'STUDENT'
mask_retired = person_df['occupation_code'] == 'RETIRED'
mask_unemployed = person_df['occupation_code'] == 'UNEMPLOYED'
mask_low = person_df['occupation_code'].isin(wage_low)
mask_wage_low_mid = person_df['occupation_code'].isin(wage_low_mid)
mask_wage_mid = person_df['occupation_code'].isin(wage_mid)
mask_wage_up_mid = person_df['occupation_code'].isin(wage_up_mid)
mask_wage_high = person_df['occupation_code'].isin(wage_high)


person_df = person_df.loc[~no_data]
person_df = person_df.loc[~self_employed]
person_df.loc[mask_student, 'income'] = 27000
person_df.loc[mask_retired, 'income'] = 55000
person_df.loc[mask_unemployed, 'income'] = 0
person_df.loc[mask_low, 'income'] = 35000 + np.random.randint(-10000, 10000, size=mask_low.sum())
person_df.loc[mask_wage_low_mid, 'income'] = 45000 + np.random.randint(-15000, 15000, size=mask_wage_low_mid.sum())
person_df.loc[mask_wage_mid, 'income'] = 80000 + np.random.randint(-20000, 20000, size=mask_wage_mid.sum())
person_df.loc[mask_wage_up_mid, 'income'] = 125000 + np.random.randint(-25000, 25000, size=mask_wage_up_mid.sum())
person_df.loc[mask_wage_high, 'income'] = 250000 + np.random.randint(-100000, 100000, size=mask_wage_high.sum())



In [349]:
kyc_business_df.set_index('customer_id', inplace=True, drop=False)

business_df = business_df.join(
    kyc_business_df[['industry_code','sales']],
    how='left'
)

business_df

,num_transactions,total_amount,avg_amount,max_amount,std_amount,num_debit,num_credit,num_abm,num_card,num_cheque,num_eft,num_emt,num_westernunion,num_wire,customer_type,industry_code,sales
customer_id,,,,,,,,,,,,,,,,,
SYNID0200000024,227,1697558.45,7478.231057,1009495.13,68137.433697,0,0,0,0,162,64,1,0,0,business,9699,181876.0
SYNID0200000050,17,17523.94,1030.820000,5595.25,1394.571891,0,0,0,3,6,8,0,0,0,business,4214,250009.0
SYNID0200000104,8,46800.31,5850.038750,39275.83,13548.688524,0,0,0,0,1,7,0,0,0,business,7215,217904.0
SYNID0200000167,35,59849.86,1709.996000,23771.15,4103.388616,0,0,0,0,1,33,0,0,1,business,8649,0.0
SYNID0200000345,22,47485.19,2158.417727,22384.05,4700.036959,0,0,0,0,5,17,0,0,0,business,9861,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SYNID0200999416,8,475.34,59.417500,241.70,80.820878,0,0,0,8,0,0,0,0,0,business,8641,NaN
SYNID0200999425,9,16480.55,1831.172222,7294.78,2422.614588,0,0,0,0,0,9,0,0,0,business,4569,0.0
SYNID0200999482,20,611931.39,30596.569500,442933.20,97883.731679,0,0,7,1,12,0,0,0,0,business,4279,0.0


In [350]:
no_data2 = (((business_df['industry_code'].isna()) | (business_df['industry_code'] == 'Other'))& (business_df['sales'].isna()))
mask_rev_med = business_df['industry_code'].isin(rev_med)
mask_rev_high = business_df['industry_code'].isin(rev_high)
mask_rev_very_high = business_df['industry_code'].isin(rev_very_high)

business_df = business_df.loc[~no_data2]
business_df.loc[mask_rev_med, 'sales'] = 125000 + np.random.randint(-75000, 75000, size=mask_rev_med.sum())
business_df.loc[mask_rev_high, 'sales'] = 600000 + np.random.randint(-400000, 400000, size=mask_rev_high.sum())
business_df.loc[mask_rev_very_high, 'sales'] = 2000000 + np.random.randint(-1000000, 1000000, size=mask_rev_very_high.sum())

business_df

,num_transactions,total_amount,avg_amount,max_amount,std_amount,num_debit,num_credit,num_abm,num_card,num_cheque,num_eft,num_emt,num_westernunion,num_wire,customer_type,industry_code,sales
customer_id,,,,,,,,,,,,,,,,,
SYNID0200000024,227,1697558.45,7478.231057,1009495.13,68137.433697,0,0,0,0,162,64,1,0,0,business,9699,51823.0
SYNID0200000050,17,17523.94,1030.820000,5595.25,1394.571891,0,0,0,3,6,8,0,0,0,business,4214,2214892.0
SYNID0200000104,8,46800.31,5850.038750,39275.83,13548.688524,0,0,0,0,1,7,0,0,0,business,7215,116838.0
SYNID0200000167,35,59849.86,1709.996000,23771.15,4103.388616,0,0,0,0,1,33,0,0,1,business,8649,184648.0
SYNID0200000345,22,47485.19,2158.417727,22384.05,4700.036959,0,0,0,0,5,17,0,0,0,business,9861,131474.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SYNID0200999416,8,475.34,59.417500,241.70,80.820878,0,0,0,8,0,0,0,0,0,business,8641,126428.0
SYNID0200999425,9,16480.55,1831.172222,7294.78,2422.614588,0,0,0,0,0,9,0,0,0,business,4569,63013.0
SYNID0200999482,20,611931.39,30596.569500,442933.20,97883.731679,0,0,7,1,12,0,0,0,0,business,4279,447583.0


In [414]:
labeled_people_df = person_df.copy()
labeled_people_df = labeled_people_df[labeled_people_df.index.isin(labeled_df['customer_id'])]
labeled_people_df.loc[:, 'std_amount'] = labeled_people_df['std_amount'].fillna(0)

label_map = labeled_df.set_index('customer_id')['label']
labeled_people_df['label'] = labeled_people_df.index.map(label_map)


X = labeled_people_df.drop(columns = ['label', 'customer_type', 'occupation_code'])
Y = labeled_people_df['label']
X_train, X_test, y_train, y_test = train_test_split(X,Y, test_size=0.3, random_state=24)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(random_state = 0).fit(X_train_scaled, y_train)


In [352]:
print(log_reg.score(X_train_scaled,y_train))
print(log_reg.score(X_test_scaled, y_test))

0.9947826086956522
0.9878542510121457


In [419]:
X_people_unlabeled = person_df.copy()
X_people_unlabeled = X_people_unlabeled.drop(columns = ['customer_type', 'occupation_code'])
X_people_unlabeled['std_amount'] = X_people_unlabeled['std_amount'].fillna(0)

X_people_scaled = scaler.transform(X_people_unlabeled)

predictions = log_reg.predict(X_people_scaled)
probabilities = log_reg.predict_proba(X_people_scaled)[:, 1]

threshold = 0.9
final_predictions = (probabilities >= threshold).astype(int)

In [354]:
# X_new = person_df.copy()
# nan_rows = X_new[X_new['income'].isna()]

# # Display them
# nan_rows

In [420]:
feature_names = X_people_unlabeled.columns.tolist()
coefficients = log_reg.coef_[0]
intercept = log_reg.intercept_[0]

contributions = X_people_scaled * coefficients
contrib_df = pd.DataFrame(contributions, columns=feature_names, index=X_people_unlabeled.index)

# Top 3 positive contributors per customer
top_pos = contrib_df.apply(lambda row: row.nlargest(3).index.tolist(), axis=1)

# Top 3 negative contributors (risk reducers)
top_neg = contrib_df.apply(lambda row: row.nsmallest(3).index.tolist(), axis=1)

# Build DataFrame with explanations
explanations_df = pd.DataFrame({
    'top_pos': top_pos,
    'top_neg': top_neg,
    'probabilities': probabilities,
})

# Create explanation text
def generate_explanation(row):
    explaination = ''
    if row['probabilities'] >= 0.5:
        explaination =  "Risk increased by: " + ", ".join(row['top_pos']) + "."
    else:
        explaination = "Risk decreased by: " + ", ".join(row['top_neg']) + "."
    return explaination


explanations_df['explanation'] = explanations_df.apply(generate_explanation, axis=1)
explanations_df['customer_id'] = X_people_unlabeled.index


In [417]:
X_people_unlabeled['prediction'] = final_predictions

unlabeled_people_output_df = X_people_unlabeled.copy()

unlabeled_people_output_df['explanation'] = explanations_df['explanation']
unlabeled_people_output_df['prediction'] = final_predictions
unlabeled_people_output_df['probabilities'] = probabilities

high_risk_customers  = unlabeled_people_output_df['prediction']

print((unlabeled_people_output_df['prediction'] == 0).sum())
print((unlabeled_people_output_df['prediction'] == 1).sum())


unlabeled_people_output_df.loc[unlabeled_people_output_df['prediction'] == 1]

49275
125


,num_transactions,total_amount,avg_amount,max_amount,std_amount,num_debit,num_credit,num_abm,num_card,num_cheque,num_eft,num_emt,num_westernunion,num_wire,income,prediction,explanation,probabilities
customer_id,,,,,,,,,,,,,,,,,,
SYNID0100019645,57,296124.38,5195.164561,205955.52,27493.579409,0,0,1,9,4,18,15,0,10,55000.0,1,"Risk increased by: total_amount, num_wire, inc...",0.995286
SYNID0100068815,412,666193.18,1616.973738,414614.62,20419.583956,0,0,1,24,0,378,9,0,0,27000.0,1,"Risk increased by: total_amount, income, num_c...",0.975847
SYNID0100118838,54,720473.69,13342.105370,482463.21,71469.709516,0,0,23,2,14,15,0,0,0,55000.0,1,"Risk increased by: total_amount, income, num_emt.",0.995674
SYNID0100264783,64,531327.85,8301.997656,320868.65,41559.725966,0,0,0,13,1,18,14,0,18,27000.0,1,"Risk increased by: total_amount, num_wire, inc...",1.000000
SYNID0100340251,77,102111.90,1326.128571,51837.57,6136.280105,0,0,0,48,0,18,3,0,8,51873.0,1,"Risk increased by: total_amount, num_wire, inc...",0.966821
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SYNID0109861268,587,566519.88,965.110528,195131.20,8088.027910,0,0,1,26,0,558,1,0,1,0.0,1,"Risk increased by: total_amount, income, num_w...",0.999850
SYNID0109876459,348,144349.28,414.796782,5991.42,709.655354,0,0,0,8,0,14,326,0,0,0.0,1,"Risk increased by: total_amount, income, num_c...",0.998327
SYNID0109878232,753,294274.53,390.802829,10158.49,765.025196,0,0,3,34,0,9,707,0,0,113947.0,1,"Risk increased by: total_amount, num_cheque, n...",1.000000


In [357]:
labeled_business_df = business_df.copy()

labeled_business_df = labeled_business_df[labeled_business_df.index.isin(labeled_df['customer_id'])]
labeled_business_df.loc[:, 'std_amount'] = labeled_business_df['std_amount'].fillna(0)

label_map2 = labeled_df.set_index('customer_id')['label']
labeled_business_df['label'] = labeled_business_df.index.map(label_map2)

X_business = labeled_business_df.drop(columns = ['label', 'customer_type', 'industry_code'])
Y_business = labeled_business_df['label']
X_business_train, X__business_test, y__business_train, y__business_test = train_test_split(X_business,Y_business, test_size=0.3, random_state=24)

scaler = StandardScaler()
X_business_train_scaled = scaler.fit_transform(X_business_train)
X_business_test_scaled = scaler.transform(X__business_test)

log_reg2 = LogisticRegression(random_state = 12).fit(X_business_train_scaled, y__business_train)

In [358]:
print(log_reg.score(X_business_train_scaled,y__business_train))
print(log_reg.score(X_business_test_scaled, y__business_test))

0.9344262295081968
1.0


In [396]:
X_business_unlabeled = business_df.copy()
X_business_unlabeled = X_business_unlabeled.drop(columns = ['customer_type', 'industry_code'])
X_business_unlabeled['std_amount'] = X_business_unlabeled['std_amount'].fillna(0)

X_business_scaled = scaler.transform(X_business_unlabeled)

predictions2 = log_reg.predict(X_business_scaled)
probabilities2 = log_reg.predict_proba(X_business_scaled)[:, 1]

threshold = 0.9
final_predictions2 = (probabilities2 >= threshold).astype(int)

In [360]:
# X_business_new = business_df.copy()
# nan_rows = X_business_new[X_business_new['sales'].isna()]

# # Display them
# nan_rows

In [ ]:
feature_names = X_business_unlabeled.columns.tolist()
coefficients = log_reg.coef_[0]
intercept = log_reg.intercept_[0]

contributions = X_business_scaled * coefficients
contrib_df = pd.DataFrame(contributions, columns=feature_names, index=X_business_unlabeled.index)

# Top 3 positive contributors per customer
top_pos = contrib_df.apply(lambda row: row.nlargest(3).index.tolist(), axis=1)

# Top 3 negative contributors (risk reducers)
top_neg = contrib_df.apply(lambda row: row.nsmallest(3).index.tolist(), axis=1)

# Build DataFrame with explanations
explanations_df2 = pd.DataFrame({
    'top_pos': top_pos,
    'top_neg': top_neg,
    'probabilities': probabilities2
})


# Create explanation text
def generate_explanation(row):
    explaination = ''
    if row['probabilities'] >= 0.5:
        explaination = "Risk increased by: " + ", ".join(row['top_pos']) + "."
    else:
        explaination =  "Risk decreased by: " + ", ".join(row['top_neg']) + "."
    return explaination

explanations_df2['explanation'] = explanations_df2.apply(generate_explanation, axis=1)
explanations_df2['customer_id'] = explanations_df2.index

In [398]:
X_business_unlabeled['prediction'] = final_predictions2

unlabeled_business_output_df = X_business_unlabeled.copy()

unlabeled_business_output_df['explanation'] = explanations_df2['explanation']
unlabeled_business_output_df['prediction'] = final_predictions2
unlabeled_business_output_df['probabilities'] = probabilities2

high_risk_businesses  = unlabeled_business_output_df['prediction']

print((unlabeled_business_output_df['prediction'] == 0).sum())
print((unlabeled_business_output_df['prediction'] == 1).sum())


unlabeled_business_output_df.loc[unlabeled_business_output_df['prediction'] == 1]

8131
23


,num_transactions,total_amount,avg_amount,max_amount,std_amount,num_debit,num_credit,num_abm,num_card,num_cheque,num_eft,num_emt,num_westernunion,num_wire,sales,prediction,explanation,probabilities
customer_id,,,,,,,,,,,,,,,,,,
SYNID0200021203,744,5290799.88,7111.290161,629004.76,40611.246791,0,0,2,2,639,99,2,0,0,197585.0,1,"Risk increased by: total_amount, sales, num_emt.",0.999642
SYNID0200045450,27,2649349.10,98124.040741,619160.26,172034.760816,0,0,0,0,0,0,0,0,27,175484.0,1,"Risk increased by: total_amount, num_wire, num...",0.994639
SYNID0200151421,1220,6800215.18,5573.946869,975084.94,35290.804689,0,0,0,8,757,455,0,0,0,151999.0,1,"Risk increased by: total_amount, sales, num_abm.",0.990449
SYNID0200155033,33,863430.09,26164.548182,204833.50,41929.787802,0,0,0,1,0,0,0,0,32,129349.0,1,"Risk increased by: num_wire, total_amount, num...",0.996822
SYNID0200159270,2273,11878999.36,5226.132582,2234930.25,55991.973087,0,0,0,0,21,2252,0,0,0,134411.0,1,"Risk increased by: total_amount, sales, num_abm.",1.000000
SYNID0200161910,2213,10326384.87,4666.238080,1594241.33,38445.634012,0,0,0,0,0,2202,11,0,0,767301.0,1,"Risk increased by: total_amount, num_cheque, n...",1.000000
SYNID0200170062,369,2418512.98,6554.235718,240003.09,21020.657273,0,0,0,0,0,341,0,0,28,63976.0,1,"Risk increased by: total_amount, num_wire, sales.",0.999999
SYNID0200192167,59,568169.15,9629.985593,107040.05,19695.253287,0,0,1,25,1,0,0,0,32,107039.0,1,"Risk increased by: num_wire, total_amount, sales.",0.997633
SYNID0200196852,764,5396089.48,7062.944346,617776.13,37320.795748,0,0,0,2,756,1,5,0,0,1220452.0,1,"Risk increased by: total_amount, num_eft, num_...",0.982620


In [ ]:
df_predictions = pd.concat([
    unlabeled_people_output_df[['prediction', 'probabilities']],
    unlabeled_business_output_df[['prediction', 'probabilities']]
])
df_predictions = df_predictions.reset_index()

df_explainations = pd.concat([
    unlabeled_people_output_df['explanation'],
    unlabeled_business_output_df['explanation']
])
df_explainations = df_explainations.reset_index()

In [421]:
df_explainations.to_csv('model_output_explainations.csv')

In [ ]:
df_predictions.to_csv('model_output.csv')